In [5]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas12-KopiNusantara-2505050006") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [8]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

# 1. StringIndexer untuk ketiga kolom kategorikal
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

# 2. OneHotEncoder untuk mengubah indeks menjadi representasi biner
ohe = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec"]
)

# 3. VectorAssembler menggabungkan fitur encoded dan fitur numerik
assembler = VectorAssembler(
    inputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

# 4. Algoritma Logistic Regression dengan target ulasan_positif
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

# Merangkai seluruh 6 tahap menjadi satu Pipeline
pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])
print("Pipeline Bagian A berhasil dibangun.")

Pipeline Bagian A berhasil dibangun.


In [9]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Membagi data 80% train dan 20% test dengan seed=77
train, test = df.randomSplit([0.8, 0.2], seed=77)
print(f"Data Train: {train.count()} baris | Data Test: {test.count()} baris")

# Melatih pipeline pada data train dan memprediksi data test
model_lr = pipeline_lr.fit(train)
predictions_lr = model_lr.transform(test)

# Mendefinisikan evaluator
ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(predictions_lr)
f1_lr = ev_f1.evaluate(predictions_lr)
auc_lr = ev_auc.evaluate(predictions_lr)

print("=== Hasil Evaluasi Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion Matrix:")
predictions_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Data Train: 2411 baris | Data Test: 589 baris
=== Hasil Evaluasi Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [10]:
from pyspark.ml.classification import RandomForestClassifier

# Mendefinisikan Random Forest dengan 30 pohon keputusan
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="ulasan_positif",
    numTrees=30,
    seed=77
)

# Menggunakan kembali 5 tahap preprocessing dari Bagian A, hanya tahap terakhir diganti
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
predictions_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(predictions_rf)
f1_rf = ev_f1.evaluate(predictions_rf)
auc_rf = ev_auc.evaluate(predictions_rf)

print("=== Perbandingan Logistic Regression vs Random Forest ===")
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | F1-score: {f1_lr:.4f} | AUC-ROC: {auc_lr:.4f}")
print(f"Random Forest       -> Accuracy: {acc_rf:.4f} | F1-score: {f1_rf:.4f} | AUC-ROC: {auc_rf:.4f}")

=== Perbandingan Logistic Regression vs Random Forest ===
Logistic Regression -> Accuracy: 0.7929 | F1-score: 0.7926 | AUC-ROC: 0.8650
Random Forest       -> Accuracy: 0.7963 | F1-score: 0.7959 | AUC-ROC: 0.8455


In [11]:
# Memilih model terbaik otomatis berdasarkan AUC-ROC tertinggi
if auc_lr >= auc_rf:
    best_model_name = "Logistic Regression"
    best_predictions = predictions_lr
else:
    best_model_name = "Random Forest"
    best_predictions = predictions_rf

print(f"Model Terbaik (AUC tertinggi): {best_model_name}\n")

# Menyaring baris di mana model salah memprediksi
errors = best_predictions.filter(col("ulasan_positif") != col("prediction"))
print(f"Total kesalahan: {errors.count()} dari {test.count()} data test")

# Menampilkan 10 contoh kesalahan
errors.select(
    "nama_menu", "cabang", "waktu_pesan", "qty",
    "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability"
).show(10, truncate=False)

# Melihat distribusi statistik waktu_tunggu_menit khusus pada data yang salah diprediksi
print("Statistik waktu_tunggu_menit pada baris yang salah diprediksi:")
errors.select("waktu_tunggu_menit").describe().show()

Model Terbaik (AUC tertinggi): Logistic Regression

Total kesalahan: 122 dari 589 data test
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414343,0.3746120824585657] |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149014,0.08861774338509865]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008387,0.7679086206099162]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016905935,0.5738454598309406]|
|Americano|Magelang|Sore       |3  |5          